# Open Sonar Datasets - Exploratory Data Analysis

This notebook analyzes the open-source sonar datasets compiled by the REMARO network.

**Source**: [OpenSonarDatasets Repository](https://github.com/remaro-network/OpenSonarDatasets)

**Paper**: M. Aubard, A. Madureira, L. Teixeira and J. Pinto, *Sonar-Based Deep Learning in Underwater Robotics: Overview, Robustness, and Challenges*, IEEE Journal of Oceanic Engineering, 2025.

---

## Dataset Overview

Analyzing 21 open-source sonar datasets covering:
- **Sonar Types**: SSS, FLS, MSIS, MBES, SAS
- **Tasks**: Classification, Object Detection, Segmentation, SLAM
- **Years**: 2010-2025

In [1]:
# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


## 1. Dataset Collection

In [2]:
# Define base path
base_path = r'C:\Users\aarav\Downloads\archive'

def count_files(directory):
    """Count all files in a directory"""
    if not os.path.exists(directory):
        return 0
    return len([f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))])

def get_file_extensions(directory):
    """Get all unique file extensions in directory"""
    if not os.path.exists(directory):
        return set()
    exts = set()
    try:
        for f in os.listdir(directory):
            if os.path.isfile(os.path.join(directory, f)):
                ext = os.path.splitext(f)[1].lower()
                if ext:
                    exts.add(ext)
    except:
        pass
    return exts

# Scan dataset structure
dataset_info = []

# Define locations and their surveys
locations = {
    'canyons': ['calibration', 'flatiron', 'horse_canyon', 'tiny_canyon', 'u_canyon'],
    'red_sea': ['big_dice_loop', 'calibration', 'coral_table_loop', 'cross_pyramid_loop', 
                'dice_path', 'landward_path', 'northeast_path', 'pier_path', 'sub_pier']
}

for location, surveys in locations.items():
    for survey in surveys:
        survey_path = os.path.join(base_path, location, survey)
        if os.path.isdir(survey_path):
            # Count files in each subdirectory
            imgs = count_files(os.path.join(survey_path, 'imgs'))
            depth = count_files(os.path.join(survey_path, 'depth'))
            seaerra = count_files(os.path.join(survey_path, 'seaErra')) + count_files(os.path.join(survey_path, 'seaerra'))
            
            # Get file types
            img_exts = get_file_extensions(os.path.join(survey_path, 'imgs'))
            depth_exts = get_file_extensions(os.path.join(survey_path, 'depth'))
            
            # Infer sonar type and data type
            sonar_type = 'MBES' if location == 'canyons' else 'SSS'
            data_type = []
            if imgs > 0:
                data_type.append('Images')
            if depth > 0:
                data_type.append('Depth Maps')
            if seaerra > 0:
                data_type.append('Error Maps')
            
            dataset_info.append({
                'name': f"{location}_{survey}",
                'location': location,
                'survey': survey,
                'sonar': sonar_type,
                'annotation': 'SLAM',
                'data_type': '+'.join(data_type) if data_type else 'Unknown',
                'num_samples': imgs,
                'depth_files': depth,
                'seaerra_files': seaerra,
                'total_files': imgs + depth + seaerra,
                'year': 2024.0,
                'has_paper': False,
                'setup': True,
                'img_formats': ','.join(sorted(img_exts)) if img_exts else 'None',
                'depth_formats': ','.join(sorted(depth_exts)) if depth_exts else 'None',
                'objects': 'Underwater Terrain',
                'completeness_score': 100 if (imgs > 0 and depth > 0 and seaerra > 0) else 
                                     66.7 if (imgs > 0 and depth > 0) or (imgs > 0 and seaerra > 0) else
                                     33.3 if imgs > 0 else 0
            })

# Create DataFrame
df = pd.DataFrame(dataset_info)

print(f"Total datasets: {len(df)}")
print(f"\nDataset columns: {list(df.columns)}")
df.head(10)

Total datasets: 0

Dataset columns: []


""


## 2. Basic Statistics

In [3]:
print("="*60)
print("SONAR DATASETS - KEY STATISTICS")
print("="*60)

print(f"\n📊 Overall Statistics:")
print(f"  Total Surveys: {len(df)}")
print(f"  Locations: {df['location'].nunique()}")
print(f"    - Canyons: {len(df[df['location']=='canyons'])}")
print(f"    - Red Sea: {len(df[df['location']=='red_sea'])}")
print(f"  Year Range: {df['year'].min():.0f} - {df['year'].max():.0f}")

print(f"\n🗂️ Data Files:")
print(f"  Total Files: {df['total_files'].sum():,}")
print(f"  Total Images: {df['num_samples'].sum():,}")
print(f"  Total Depth Maps: {df['depth_files'].sum():,}")
print(f"  Total Error Maps: {df['seaerra_files'].sum():,}")

print(f"\n📈 Sonar Types Distribution:")
for sonar_type, count in df['sonar'].value_counts().items():
    print(f"  {sonar_type}: {count} surveys")

print(f"\n🎯 Annotation Types:")
for ann_type, count in df['annotation'].value_counts().items():
    print(f"  {ann_type}: {count} surveys")

print(f"\n💾 File Formats:")
all_img_formats = set()
all_depth_formats = set()
for formats in df['img_formats'].dropna():
    if formats != 'None':
        all_img_formats.update(formats.split(','))
for formats in df['depth_formats'].dropna():
    if formats != 'None':
        all_depth_formats.update(formats.split(','))
print(f"  Image formats: {', '.join(sorted(all_img_formats)) if all_img_formats else 'None found'}")
print(f"  Depth formats: {', '.join(sorted(all_depth_formats)) if all_depth_formats else 'None found'}")

SONAR DATASETS - KEY STATISTICS

📊 Overall Statistics:
  Total Surveys: 0


KeyError: 'location'

## 3. Sonar Type Analysis

In [ ]:
# Sonar type distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Sonar types
sonar_counts = df['sonar'].value_counts()
axes[0].bar(range(len(sonar_counts)), sonar_counts.values, color='steelblue')
axes[0].set_xticks(range(len(sonar_counts)))
axes[0].set_xticklabels(sonar_counts.index, rotation=0)
axes[0].set_ylabel('Number of Surveys', fontsize=11)
axes[0].set_title('Distribution by Sonar Type', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

# Add counts on bars
for i, v in enumerate(sonar_counts.values):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# 2. Location distribution
location_counts = df['location'].value_counts()
colors = ['#2E86AB', '#A23B72']
axes[1].bar(range(len(location_counts)), location_counts.values, color=colors)
axes[1].set_xticks(range(len(location_counts)))
axes[1].set_xticklabels([loc.replace('_', ' ').title() for loc in location_counts.index], rotation=0)
axes[1].set_ylabel('Number of Surveys', fontsize=11)
axes[1].set_title('Distribution by Location', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Add counts on bars
for i, v in enumerate(location_counts.values):
    axes[1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nSonar Type Breakdown:")
print(df.groupby('sonar')['num_samples'].agg(['count', 'sum', 'mean']).round(0))

## 4. Annotation Task Analysis

In [ ]:
# Annotation types and data characteristics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Samples by location
location_samples = df.groupby('location')['num_samples'].sum()
axes[0].barh(range(len(location_samples)), location_samples.values, color=['#2E86AB', '#A23B72'])
axes[0].set_yticks(range(len(location_samples)))
axes[0].set_yticklabels([loc.replace('_', ' ').title() for loc in location_samples.index])
axes[0].set_xlabel('Total Images', fontsize=11)
axes[0].set_title('Total Images by Location', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add values
for i, v in enumerate(location_samples.values):
    axes[0].text(v + max(location_samples.values)*0.01, i, f'{v:,.0f}', va='center')

# 2. File type composition
file_types = df[['location', 'num_samples', 'depth_files', 'seaerra_files']].groupby('location').sum()
file_types.plot(kind='bar', stacked=False, ax=axes[1], color=['#F18F01', '#2E86AB', '#A23B72'])
axes[1].set_xlabel('Location', fontsize=11)
axes[1].set_ylabel('File Count', fontsize=11)
axes[1].set_title('File Type Distribution by Location', fontsize=13, fontweight='bold')
axes[1].set_xticklabels([loc.replace('_', ' ').title() for loc in file_types.index], rotation=0)
axes[1].legend(['Images', 'Depth Files', 'Error Files'], loc='upper right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAnnotation Type Statistics:")
print(df.groupby('annotation')[['num_samples', 'total_files']].agg(['count', 'sum', 'mean']).round(1))

## 5. Dataset Sizes

In [ ]:
# Dataset sizes analysis
df_with_samples = df[df['num_samples'] > 0].copy()
df_with_samples = df_with_samples.sort_values('num_samples', ascending=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top datasets by size
top_10 = df_with_samples.head(10)
colors_by_location = ['#2E86AB' if loc == 'canyons' else '#A23B72' for loc in top_10['location']]
axes[0].barh(range(len(top_10)), top_10['num_samples'], color=colors_by_location)
axes[0].set_yticks(range(len(top_10)))
axes[0].set_yticklabels([name.replace('_', ' ').title() for name in top_10['name']])
axes[0].set_xlabel('Number of Images', fontsize=12)
axes[0].set_title('Top 10 Surveys by Image Count', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add values on bars
for i, v in enumerate(top_10['num_samples']):
    axes[0].text(v + max(top_10['num_samples'])*0.01, i, f'{v:,.0f}', va='center', fontsize=9)

# Distribution of dataset sizes
if len(df_with_samples) > 0 and df_with_samples['num_samples'].max() > 0:
    sample_sizes = df_with_samples['num_samples']
    sample_sizes = sample_sizes[sample_sizes > 0]
    
    if len(sample_sizes) > 0:
        axes[1].hist(np.log10(sample_sizes), bins=15, 
                    color='skyblue', edgecolor='black', alpha=0.7)
        axes[1].set_xlabel('log10(Number of Images)', fontsize=12)
        axes[1].set_ylabel('Frequency', fontsize=12)
        axes[1].set_title('Distribution of Survey Sizes (Log Scale)', fontsize=14, fontweight='bold')
        axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Size categories
print("\n" + "="*60)
print("DATASET SIZE CATEGORIES")
print("="*60)
if len(df_with_samples) > 0:
    small = len(df_with_samples[df_with_samples['num_samples'] < 100])
    medium = len(df_with_samples[(df_with_samples['num_samples'] >= 100) & (df_with_samples['num_samples'] < 1000)])
    large = len(df_with_samples[(df_with_samples['num_samples'] >= 1000) & (df_with_samples['num_samples'] < 5000)])
    xlarge = len(df_with_samples[df_with_samples['num_samples'] >= 5000])
    
    print(f"  Small (<100 images):     {small} surveys")
    print(f"  Medium (100-1K):         {medium} surveys")
    print(f"  Large (1K-5K):           {large} surveys")
    print(f"  Extra Large (>5K):       {xlarge} surveys")

## 6. Object Label Analysis

In [ ]:
# Object/terrain type analysis
print("="*60)
print("TERRAIN/OBJECT ANALYSIS")
print("="*60)

# For SLAM datasets, categorize surveys by type
survey_types = {}
for idx, row in df.iterrows():
    survey_name = row['survey'].lower()
    # Categorize surveys by name
    if 'calibration' in survey_name:
        category = 'Calibration'
    elif 'canyon' in survey_name:
        category = 'Canyon Terrain'
    elif 'loop' in survey_name:
        category = 'Loop Path'
    elif 'path' in survey_name:
        category = 'Path Navigation'
    elif 'pier' in survey_name:
        category = 'Pier Structure'
    else:
        category = 'Other'
    
    if category not in survey_types:
        survey_types[category] = []
    survey_types[category].append(row['name'])

print(f"\nSurvey Type Distribution:")
print(f"Total survey categories: {len(survey_types)}\n")

for category, surveys in sorted(survey_types.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {category:25s} - {len(surveys)} survey(s)")
    for survey in surveys[:3]:
        print(f"    • {survey}")
    if len(surveys) > 3:
        print(f"    ... and {len(surveys)-3} more")
    print()

# Visualize survey categories
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
categories = list(survey_types.keys())
counts = [len(surveys) for surveys in survey_types.values()]
ax.barh(categories, counts, color='teal')
ax.set_xlabel('Number of Surveys', fontsize=12)
ax.set_title('Survey Type Distribution', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for i, v in enumerate(counts):
    ax.text(v + 0.1, i, str(v), va='center')

plt.tight_layout()
plt.show()

## 7. Temporal Trends

In [ ]:
# Temporal analysis
print("="*60)
print("TEMPORAL ANALYSIS")
print("="*60)

print(f"\nDataset Collection Period:")
print(f"  All surveys collected: ~{df['year'].mode()[0]:.0f}")
print(f"  Total surveys: {len(df)}")

# Analyze by location
location_timeline = df.groupby('location').agg({
    'num_samples': 'sum',
    'total_files': 'sum',
    'survey': 'count'
}).rename(columns={'survey': 'num_surveys'})

print(f"\nData Collection by Location:")
print(location_timeline)

# Timeline visualization
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for idx, location in enumerate(df['location'].unique()):
    location_df = df[df['location'] == location].sort_values('survey')
    x_vals = range(len(location_df))
    y_vals = [idx] * len(location_df)
    sizes = location_df['num_samples'].values / 50
    
    ax.scatter(x_vals, y_vals, s=sizes, alpha=0.6, 
              label=location.replace('_', ' ').title(),
              c='#2E86AB' if location == 'canyons' else '#A23B72')

ax.set_yticks(range(len(df['location'].unique())))
ax.set_yticklabels([loc.replace('_', ' ').title() for loc in df['location'].unique()])
ax.set_xlabel('Survey Index', fontsize=12)
ax.set_title('Survey Distribution by Location (size = image count)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Data Type Analysis

In [ ]:
# Data type and format analysis
print("="*60)
print("DATA TYPE & FORMAT ANALYSIS")
print("="*60)

print(f"\nData Type Distribution:")
data_type_counts = df['data_type'].value_counts()
for dt, count in data_type_counts.items():
    print(f"  {dt}: {count} surveys")

print(f"\nFile Completeness:")
print(f"  Surveys with images: {(df['num_samples'] > 0).sum()}/{len(df)}")
print(f"  Surveys with depth data: {(df['depth_files'] > 0).sum()}/{len(df)}")
print(f"  Surveys with error maps: {(df['seaerra_files'] > 0).sum()}/{len(df)}")
print(f"  Complete surveys (all 3): {((df['num_samples'] > 0) & (df['depth_files'] > 0) & (df['seaerra_files'] > 0)).sum()}/{len(df)}")

# Visualize data completeness
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Stacked bar of file types per survey
top_surveys = df.nlargest(10, 'total_files')
survey_names = [name.replace('_', '\n') for name in top_surveys['name']]
x = np.arange(len(top_surveys))
width = 0.6

p1 = axes[0].bar(x, top_surveys['num_samples'], width, label='Images', color='#F18F01')
p2 = axes[0].bar(x, top_surveys['depth_files'], width, bottom=top_surveys['num_samples'], 
                label='Depth', color='#2E86AB')
p3 = axes[0].bar(x, top_surveys['seaerra_files'], width, 
                bottom=top_surveys['num_samples'] + top_surveys['depth_files'],
                label='Error', color='#A23B72')

axes[0].set_ylabel('File Count', fontsize=11)
axes[0].set_title('File Composition - Top 10 Surveys', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(survey_names, rotation=45, ha='right', fontsize=8)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 2. Completeness pie chart
completeness = {
    'Complete\n(All 3 types)': ((df['num_samples'] > 0) & (df['depth_files'] > 0) & (df['seaerra_files'] > 0)).sum(),
    'Partial\n(1-2 types)': ((df['num_samples'] > 0) | (df['depth_files'] > 0) | (df['seaerra_files'] > 0)).sum() - 
                            ((df['num_samples'] > 0) & (df['depth_files'] > 0) & (df['seaerra_files'] > 0)).sum(),
    'Empty': ((df['num_samples'] == 0) & (df['depth_files'] == 0) & (df['seaerra_files'] == 0)).sum()
}
completeness = {k: v for k, v in completeness.items() if v > 0}

axes[1].pie(completeness.values(), labels=completeness.keys(), autopct='%1.1f%%',
           colors=['#2E86AB', '#F18F01', '#A23B72'], startangle=90)
axes[1].set_title('Survey Completeness', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Quality & Completeness Assessment

In [ ]:
# Quality and completeness metrics
print("="*60)
print("DATASET QUALITY & COMPLETENESS")
print("="*60)

# Calculate quality metrics
df['has_images'] = df['num_samples'] > 0
df['has_depth'] = df['depth_files'] > 0
df['has_error'] = df['seaerra_files'] > 0
df['file_diversity'] = df['has_images'].astype(int) + df['has_depth'].astype(int) + df['has_error'].astype(int)

print(f"\nCompleteness Metrics:")
print(f"  Average completeness score: {df['completeness_score'].mean():.1f}/100")
print(f"  Surveys with 100% completeness: {(df['completeness_score'] == 100).sum()}/{len(df)}")
print(f"  Average file diversity: {df['file_diversity'].mean():.1f}/3 file types")

print(f"\nQuality Indicators:")
print(f"  Surveys with setup documentation: {df['setup'].sum()}/{len(df)}")
print(f"  Surveys with papers: {df['has_paper'].sum()}/{len(df)}")
print(f"  Average files per survey: {df['total_files'].mean():.0f}")

# Quality score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Completeness score distribution
completeness_bins = [0, 33, 67, 100]
completeness_labels = ['Low\n(0-33%)', 'Medium\n(34-67%)', 'High\n(68-100%)']
df['completeness_category'] = pd.cut(df['completeness_score'], bins=completeness_bins, 
                                      labels=completeness_labels, include_lowest=True)
completeness_counts = df['completeness_category'].value_counts().sort_index()

axes[0].bar(range(len(completeness_counts)), completeness_counts.values, 
           color=['#A23B72', '#F18F01', '#2E86AB'])
axes[0].set_xticks(range(len(completeness_counts)))
axes[0].set_xticklabels(completeness_counts.index, rotation=0)
axes[0].set_ylabel('Number of Surveys', fontsize=11)
axes[0].set_title('Completeness Score Distribution', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(completeness_counts.values):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# 2. Completeness by location
location_completeness = df.groupby('location')['completeness_score'].mean()
axes[1].barh(range(len(location_completeness)), location_completeness.values,
            color=['#2E86AB', '#A23B72'])
axes[1].set_yticks(range(len(location_completeness)))
axes[1].set_yticklabels([loc.replace('_', ' ').title() for loc in location_completeness.index])
axes[1].set_xlabel('Average Completeness Score', fontsize=11)
axes[1].set_title('Average Completeness by Location', fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 100)
axes[1].grid(axis='x', alpha=0.3)

for i, v in enumerate(location_completeness.values):
    axes[1].text(v + 2, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

# Detailed completeness report
print(f"\n{'='*60}")
print("DETAILED COMPLETENESS REPORT")
print(f"{'='*60}")
incomplete = df[df['completeness_score'] < 100][['name', 'location', 'has_images', 'has_depth', 'has_error', 'completeness_score']]
if len(incomplete) > 0:
    print(f"\n⚠️ Incomplete Surveys ({len(incomplete)}):")
    for idx, row in incomplete.iterrows():
        print(f"  • {row['name']}")
        missing = []
        if not row['has_images']:
            missing.append('images')
        if not row['has_depth']:
            missing.append('depth')
        if not row['has_error']:
            missing.append('error maps')
        print(f"    Missing: {', '.join(missing)}")
        print(f"    Completeness: {row['completeness_score']:.0f}%")
else:
    print("\n All surveys are 100% complete!")

## 10. Summary & Recommendations

In [ ]:
# Comprehensive summary and recommendations
print("="*80)
print("COMPREHENSIVE DATASET SUMMARY")
print("="*80)

print("\n1. DATASET OVERVIEW")
print(f"   - Total surveys analyzed: {len(df)}")
print(f"   - Locations: {', '.join(df['location'].unique())}")
print(f"   - Primary sonar types: {', '.join(df['sonar'].unique())}")
print(f"   - Main task: {df['annotation'].mode()[0]}")

print("\n2. DATA VOLUME")
print(f"   - Total files: {df['total_files'].sum():,}")
print(f"   - Total images: {df['num_samples'].sum():,}")
print(f"   - Total depth maps: {df['depth_files'].sum():,}")
print(f"   - Total error maps: {df['seaerra_files'].sum():,}")
print(f"   - Average files per survey: {df['total_files'].mean():.0f}")

print("\n3. DATA DISTRIBUTION")
canyons_total = df[df['location']=='canyons']['num_samples'].sum()
red_sea_total = df[df['location']=='red_sea']['num_samples'].sum()
total = canyons_total + red_sea_total
if total > 0:
    print(f"   - Canyons: {canyons_total:,} images ({canyons_total/total*100:.1f}%)")
    print(f"   - Red Sea: {red_sea_total:,} images ({red_sea_total/total*100:.1f}%)")

print("\n4. DATASET CHARACTERISTICS")
print(f"   - Largest survey: {df.loc[df['num_samples'].idxmax(), 'name']} ({df['num_samples'].max():,} images)")
print(f"   - Smallest survey: {df.loc[df['num_samples'].idxmin(), 'name']} ({df['num_samples'].min():,} images)")
print(f"   - Median survey size: {df['num_samples'].median():.0f} images")

print("\n5. DATASET MATURITY")
print(f"   - {df['setup'].sum()}/{len(df)} surveys have complete directory structure")
print(f"   - Average completeness score: {df['completeness_score'].mean():.1f}/100")
print(f"   - Surveys with all file types: {(df['completeness_score']==100).sum()}/{len(df)}")

print("\n" + "="*80)
print("💡 RECOMMENDATIONS FOR ANALYSIS")
print("="*80)

print("\n📌 FOR SLAM RESEARCH:")
complete_slam = df[df['completeness_score'] == 100].nlargest(3, 'num_samples')
if len(complete_slam) > 0:
    print("   Best complete datasets for SLAM:")
    for idx, row in complete_slam.iterrows():
        print(f"   - {row['name']} ({row['num_samples']:,} images, {row['total_files']:,} total files)")

print("\n📌 FOR DEPTH MAPPING:")
best_depth = df[df['depth_files'] > 0].nlargest(3, 'depth_files')
if len(best_depth) > 0:
    print("   Surveys with most depth data:")
    for idx, row in best_depth.iterrows():
        print(f"   - {row['name']} ({row['depth_files']:,} depth files)")

print("\n📌 FOR ERROR ANALYSIS:")
best_error = df[df['seaerra_files'] > 0].nlargest(3, 'seaerra_files')
if len(best_error) > 0:
    print("   Surveys with most error/uncertainty data:")
    for idx, row in best_error.iterrows():
        print(f"   - {row['name']} ({row['seaerra_files']:,} error files)")

print("\n📌 BY LOCATION:")
print("   Canyons surveys (MBES):")
canyon_surveys = df[df['location']=='canyons'].nlargest(3, 'num_samples')
for idx, row in canyon_surveys.iterrows():
    print(f"   - {row['survey']} ({row['num_samples']:,} images)")

print("\n   Red Sea surveys (SSS):")
red_sea_surveys = df[df['location']=='red_sea'].nlargest(3, 'num_samples')
for idx, row in red_sea_surveys.iterrows():
    print(f"   - {row['survey']} ({row['num_samples']:,} images)")

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE")
print("="*80)
print(f"\nDataset: Local Sonar Archive")
print(f"Location: {base_path}")
print(f"\nAnalyzed {len(df)} surveys from {df['location'].nunique()} locations.")
print(f"Total data volume: {df['total_files'].sum():,} files")

## 11. Export Summary

In [ ]:
# Create summary dataframe
summary_df = df[['name', 'location', 'survey', 'sonar', 'annotation', 'num_samples', 
                 'depth_files', 'seaerra_files', 'total_files', 'completeness_score']].copy()
summary_df = summary_df.sort_values('completeness_score', ascending=False)

# Save to CSV
output_file = 'sonar_datasets_summary.csv'
summary_df.to_csv(output_file, index=False)
print(f"✅ Summary exported to: {output_file}")

# Display top datasets
print("\n📊 Top 10 Surveys by Completeness Score:")
summary_df.head(10)